# Preparing Data

In [0]:
def processData(data):
    # print(data)
    X = data.iloc[:,:].values
    X = np.delete(X, 0, 1)
    X = np.delete(X, 1, 1)
    X = np.mean(X, 0)
    X = np.array(X.reshape(1, -1))
    # print(X)
    # print(X.shape)
    return X

def getData(filename):
    # print(filename)
    data = pd.read_csv(filename,delimiter=',')
    X = processData(data)
    return X

def makeDataPoint(patientNo):
    file1 = (patientNo)+"_CLNF_AUs.txt"
    file2 = (patientNo)+"_CLNF_features.txt"
    file3 = (patientNo)+"_CLNF_features3D.txt"
    file4 = (patientNo)+"_CLNF_gaze.txt"
    file5 = (patientNo)+"_CLNF_hog.txt"
    file6 = (patientNo)+"_CLNF_pose.txt"

    X1 = getData(file1)
    X2 = getData(file2)
    X3 = getData(file3)
    X4 = getData(file4)
    # X5 = getData(file5)
    X6 = getData(file6)

    X = np.concatenate((X1, X2, X3, X4, X6), 1)
    return X


dataset1 = np.array(pd.read_csv('/content/drive/My Drive/MCA Dataset/dev_split_Depression_AVEC2017.csv',delimiter=',',encoding='utf-8'))[:, 0:2]
dataset2 = np.array(pd.read_csv('/content/drive/My Drive/MCA Dataset/full_test_split.csv',delimiter=',',encoding='utf-8'))[:, 0:2]
dataset3 = np.array(pd.read_csv('/content/drive/My Drive/MCA Dataset/train_split_Depression_AVEC2017.csv',delimiter=',',encoding='utf-8'))[:, 0:2]

dataset = np.concatenate((dataset1, np.concatenate((dataset2, dataset3))))
positive = []
negative = []

for i in range(len(dataset)):
    if dataset[i][0]<=310 and dataset[i][0]>=300:
        if(dataset[i][1] == 0):
            positive.append(int(dataset[i][0]))
        else:
            negative.append(int(dataset[i][0]))

    elif dataset[i][0]<=355 and dataset[i][0]>=345:
        if(dataset[i][1] == 0):
            positive.append(int(dataset[i][0]))
        else:
            negative.append(int(dataset[i][0]))

X = np.zeros((1,388))
for i in positive:
    string = '/content/drive/My Drive/MCA Dataset/positive_data/'+str(i)
    X_temp = makeDataPoint(string)
    X = np.concatenate((X,X_temp),0)
Y = np.zeros((len(positive),1))
for i in negative:
    string = '/content/drive/My Drive/MCA Dataset/negative_data/'+str(i)
    X_temp = makeDataPoint(string)
    X = np.concatenate((X,X_temp),0)
Y = np.concatenate((Y,np.ones((len(negative),1))),0)
X = np.delete(X,0,0)

# print("train")
X_train, Y_train = makeDataset('/content/drive/My Drive/MCA Dataset/train_split_Depression_AVEC2017.csv', 'train_data')
# print("dev")
X_dev, Y_dev = makeDataset('/content/drive/My Drive/MCA Dataset/dev_split_Depression_AVEC2017.csv', 'dev_data')
count0 = 0
X_train = np.concatenate((X_train, X_dev), 0  )
Y_train = np.concatenate((Y_train, Y_dev), 0  )

# print("test")
X_test, Y_test = makeDataset('/content/drive/My Drive/MCA Dataset/full_test_split.csv', 'test_data')
# print(Y_train)

**RF and SVM on dataset**

In [0]:

clf1 = RandomForestClassifier()
clf1.fit(X_train,Y_train)
clf1.predict(X_test)
print("Random forest: "+ str(clf1.score(X_test,Y_test)))


classifier2 = SVC(kernel = 'rbf', random_state = 0)
classifier2.fit(X_train, Y_train)
classifier2.predict(X_test)
print("SVM rbf: "+ str(classifier2.score(X_test,Y_test)))

clf1.predict(X_train)
print("Random forest: train"+ str(clf1.score(X_train,Y_train)))

classifier2.predict(X_train)
print("SVM: train  "+ str(classifier2.score(X_train,Y_train)))


y_pred = classifier2.predict(X_test)
print(classification_report(Y_test, y_pred))

In [0]:
class RfVideo:
  def __init__(self):
    self.classifier = RandomForestClassifier()
  
  def fitModel(self, X, Y):
    self.classfier.fit(X, Y)
  
  def predictModel(self, X):
    return self.classifier.predict(X)
  
  def scoreModel(self, X, Y):
    return self.classifier.score(X, Y)

class SVMVideo:
  def __init__(self, kernel = "rbf"):
    self.classifier = SVC(kernel = kernel, random_state = 0)
  
  def fitModel(self, X, Y):
    self.classfier.fit(X, Y)
  
  def predictModel(self, X):
    return self.classifier.predict(X)
  
  def scoreModel(self, X, Y):
    return self.classifier.score(X, Y)

print("Random forest: ")
rfModel = RfVideo()
rfModel.fitModel(X_train, Y_train)
Y_pred1  = rfModel.predict(X_test)
print(classification_report(Y_test, Y_pred1))

print("SVM: ")
svmModel = SVMVideo()
svmModel.fitModel(X_train, Y_train)
Y_pred2  = svmModel.predict(X_test)
print(classification_report(Y_test, Y_pred2))